# 1. Nominal Categorical Encoding: One-Hot Encoding & Dummy Variables

This notebook covers:
1. Converting nominal categorical features into numeric binary vectors via **One-Hot Encoding**.
2. The **Dummy Variable Trap** (Multicollinearity) and when to use `drop='first'`.
3. Handling **Unseen / Unknown categories** in test data and production environments (`handle_unknown='ignore'`).
4. Comparing `pd.get_dummies()` vs. Scikit-Learn's `OneHotEncoder` for production workflows.

In [27]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# Create a sample raw dataset with nominal features
data = {
    'City': ['Hyderabad', 'Bangalore', 'Mumbai', 'Hyderabad', 'Bangalore'],
    'Device_Type': ['Android', 'iOS', 'Android', 'Windows', 'iOS'],
    'Purchased': [1, 0, 1, 0, 1]
}

df = pd.DataFrame(data)
print("=== RAW NOMINAL DATA ===")
display(df)

=== RAW NOMINAL DATA ===


,City,Device_Type,Purchased
0,Hyderabad,Android,1
1,Bangalore,iOS,0
2,Mumbai,Android,1
3,Hyderabad,Windows,0
4,Bangalore,iOS,1


---
## Part 1: How One-Hot Encoding Works

One-Hot Encoding creates a new binary column ($0$ or $1$) for every unique category present in the feature.

* If a record belongs to a category, its corresponding column receives a **`1`**; all other category columns receive a **`0`**.
* This eliminates the risk of models assuming false numerical hierarchy (e.g., $3 > 1$).

### Key Parameter: `sparse_output=False`
By default, Scikit-Learn returns a memory-efficient sparse matrix. Setting `sparse_output=False` (or `sparse=False` in older versions) returns a standard NumPy 2D array.

In [30]:
from sklearn.compose import ColumnTransformer

dfc = df.copy()

ct = ColumnTransformer(transformers=[
    ('city_encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['City']) 
    ], remainder='passthrough') 
# to remove those terms 'remainder' in front of column names in the output, add another parameter beside the remainder parameter which is: "verbose_feature_name_out"
ct.set_output(transform='pandas')

df_final = ct.fit_transform(dfc)
display(df_final)



,city_encoder__City_Bangalore,city_encoder__City_Hyderabad,city_encoder__City_Mumbai,remainder__Device_Type,remainder__Purchased
0,0.0,1.0,0.0,Android,1
1,1.0,0.0,0.0,iOS,0
2,0.0,0.0,1.0,Android,1
3,0.0,1.0,0.0,Windows,0
4,1.0,0.0,0.0,iOS,1


### code explanation


**1. `df_copy = df.copy()`**
- Creating a clean copy guarantees that original raw data remains untouched, preventing unintended in-place modifications during experimentation.

**2. `transformers=[('city_encoder', OneHotEncoder(...), ['City'])]`**
- This is a 3-element tuple: `(step_name, transformer_function, column_list)`.
- It tells Scikit-Learn: _"Isolate only the `'City'` column, convert its text categories into $0/1$ binary columns, and safely ignore unseen categories in production with `handle_unknown='ignore'`."_

**3. `remainder='passthrough'`**
- By default, `ColumnTransformer` drops any column not listed in `transformers`.
- Specifying `remainder='passthrough'` instructs it to **keep every other column** (`Age`, `Salary`, `Purchased`) untouched and stitch them right next to the new binary columns.

**4. `ct.set_output(transform='pandas')`**
- Standard Scikit-Learn outputs raw NumPy arrays without column names. This line forces `ColumnTransformer` to return a fully formatted Pandas DataFrame with clean column headers like `city_encoder__City_Hyderabad` and `remainder__Age`.

**5. `df_final = ct.fit_transform(df_copy)`**
- Executes the transformation on the full DataFrame in a single step—removing the original `'City'` text column, inserting the encoded binary columns, and preserving the rest of the table.